# Lesson 15 · Diffraction gratings and optics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/15_diffraction_gratings_and_optics.ipynb)

**For:** A-level Physics (for example AQA 3.3.2.2 and 3.3.2.3, OCR A 4.4, Edexcel Topic 5) and IB Physics (C.3 wave phenomena). The thin-lens section is extension material (it appears in the AQA Astrophysics option). Check your own specification for the exact wording.

**Before this lesson:** filled in by `scripts/lesson_path.py`.

**Leads to:** filled in by `scripts/lesson_path.py`.

**You will be able to:**

1. use the grating equation $d\sin\theta = n\lambda$, with $d$ found from the number of lines per metre
2. find the highest visible order and the total number of maxima from a grating
3. explain why more slits give sharper, brighter maxima, and why white-light spectra overlap
4. describe single-slit diffraction and use $\sin\theta = \lambda/a$ for the first minimum
5. use Snell's law $n_1\sin\theta_1 = n_2\sin\theta_2$, find a critical angle and explain total internal reflection in optical fibres
6. use the thin-lens equation $1/f = 1/u + 1/v$ to locate real and virtual images (extension)

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Install the course API in a fresh Colab runtime or upgrade an older installation.
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("qplay") is None or int(version("quantum-playground").split(".")[0]) < 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantum-playground>=1,<2"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import waves

## 1. The diffraction grating

A diffraction grating is a plate ruled with a large number of equally spaced slits. If it has $N$ lines per metre, the slit spacing is $d = 1/N$. Light from neighbouring slits arrives at a distant screen in phase whenever their **path difference** $d\sin\theta$ is a whole number of wavelengths, which gives the **grating equation**

$$d\sin\theta = n\lambda, \qquad n = 0, 1, 2, \dots$$

The whole number $n$ is the **order** of the maximum. The straight-through beam is the zero order. Because $\sin\theta$ can never exceed 1, the highest order you can see is the largest whole number with $n \le d/\lambda$. Including the zero order and both sides, the total number of maxima is $2n_{\max} + 1$.

`waves.grating_angle(n, wavelength, spacing)` returns the angle of order $n$ in radians, or `None` if that order does not exist.

A double slit obeys the same equation. So what does a grating with hundreds of slits gain?

**Predict (P1).** Keep the slit spacing fixed and increase the number of illuminated slits from 2 to 5 to 20. What happens to the **positions** of the bright maxima? What happens to their **widths**? Write your predictions down before you run the next cell.

In [ ]:
wavelength, d, a = 600e-9, 3.0e-6, 0.8e-6     # m: light, slit spacing, width of each slit
theta = np.radians(np.linspace(-45, 45, 4001))
phi = 2 * np.pi * d * np.sin(theta) / wavelength  # phase difference between neighbouring slits

def n_slit(N):
    # [sin(N phi/2) / sin(phi/2)]^2 / N^2, which tends to 1 where sin(phi/2) = 0
    top, bottom = np.sin(N * phi / 2), np.sin(phi / 2)
    safe = np.where(np.abs(bottom) < 1e-12, 1.0, bottom)
    return np.where(np.abs(bottom) < 1e-12, 1.0, (top / (N * safe)) ** 2)

fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for ax, N in zip(axes, (2, 5, 20)):
    intensity = n_slit(N) * waves.slit_envelope(theta, a, wavelength)
    ax.plot(np.degrees(theta), intensity, lw=1)
    ax.set_ylabel(f"N = {N}")
    peaks = [waves.grating_angle(n, wavelength, d) for n in range(-5, 6)]
    for peak in peaks:
        if peak is not None and abs(np.degrees(peak)) <= 45:
            ax.axvline(np.degrees(peak), color="C3", lw=0.6, ls=":")
    half = intensity > 0.5 * intensity.max()          # width of the central maximum at half height
    print(f"N = {N:2d}: central maximum is {np.ptp(np.degrees(theta[half & (np.abs(theta) < 0.1)])):.2f} degrees wide at half height")
axes[-1].set_xlabel(r"angle $\theta$ / degrees"); axes[-1].set_xlim(-45, 45)
fig.suptitle("Relative intensity behind N slits (dotted lines: $d\\sin\\theta = n\\lambda$)");

**Explain (P1).** The positions come from $d\sin\theta = n\lambda$. Why does that equation not depend on $N$? Why does adding slits make the maxima narrower? (Think about how far you must move off a maximum before the light from the first and last slits is half a wavelength out of step.)

The overall shape follows the single-slit pattern of each slit (section 3), which is why the higher orders are dimmer.

## 2. White light and overlapping spectra

Each wavelength has its own angle, so a grating spreads white light into a **spectrum** in every order except zero (where all colours go straight through and the beam stays white). Violet, with the shortest wavelength, is deviated least; red the most. Higher orders are spread out more.

**Predict (P2).** White light (400 nm to 700 nm) falls on a grating with 300 lines per mm. Do the second-order and third-order spectra overlap?

In [ ]:
d = 1e-3 / 300                                 # 300 lines per mm
colours = np.linspace(400e-9, 700e-9, 61)
plt.figure(figsize=(8, 2.8))
for n in (1, 2, 3):
    angles = np.degrees([waves.grating_angle(n, lam, d) for lam in colours])
    plt.scatter(angles, np.full_like(angles, n), c=colours * 1e9, cmap="turbo", vmin=380, vmax=720, s=40, marker="|")
    print(f"order {n}: violet at {angles[0]:5.2f} degrees, red at {angles[-1]:5.2f} degrees")
plt.yticks([1, 2, 3]); plt.ylim(0.5, 3.5); plt.ylabel("order n"); plt.xlabel(r"angle $\theta$ / degrees")
plt.title("Spectra from a 300 lines/mm grating, 400-700 nm");

**Explain (P2).** Compare the angle of second-order red with third-order violet. Which wavelength in the third order lands on top of second-order red?

## 3. Diffraction at a single slit

Light passing through one slit of width $a$ also spreads out. Different parts of the same slit interfere, and the intensity falls to zero at the first minimum, where

$$\sin\theta = \frac{\lambda}{a}.$$

The central maximum stretches from $-\theta$ to $+\theta$, so it is **twice as wide** as the other maxima, and much brighter. Narrower slits and longer wavelengths spread the light more. `waves.slit_envelope(theta, width, wavelength)` gives the intensity relative to the centre.

In [ ]:
wavelength = 500e-9
theta = np.radians(np.linspace(-30, 30, 3001))
plt.figure(figsize=(8, 3.4))
for i, a in enumerate((2e-6, 5e-6, 10e-6)):
    plt.plot(np.degrees(theta), waves.slit_envelope(theta, a, wavelength), color=f"C{i}", label=f"a = {a * 1e6:.0f} μm")
    first = np.degrees(np.arcsin(wavelength / a))
    plt.axvline(first, color=f"C{i}", ls=":", lw=0.8)
    print(f"a = {a * 1e6:4.1f} μm: first minimum at {first:5.2f} degrees")
plt.xlabel(r"angle $\theta$ / degrees"); plt.ylabel(r"$I/I_0$"); plt.legend()
plt.title("Single slit, 500 nm light (dotted: $\\sin\\theta = \\lambda/a$)");

*Try it:* change `wavelength` to 700e-9 (red). Which way do the minima move?

## 4. Refraction and total internal reflection

When light crosses a boundary its speed changes, and it bends. The **refractive index** of a material is $n = c/v$. At a boundary, **Snell's law** links the angles measured from the **normal**:

$$n_1\sin\theta_1 = n_2\sin\theta_2.$$

Going into a denser medium the ray bends towards the normal. Going from glass into air it bends away, and for a large enough angle of incidence there is no refracted ray at all: the light is **totally internally reflected**. The boundary case, where the refracted ray just skims along the surface ($\theta_2 = 90^\circ$), is the **critical angle** $C$:

$$\sin C = \frac{n_2}{n_1}.$$

`waves.refraction(theta, n1, n2)` returns the refracted angle, or `None` when there is no refracted ray.

**Predict (P3).** Light travels inside glass ($n = 1.5$) towards a boundary with air. Which of these angles of incidence gives total internal reflection: $30^\circ$ or $50^\circ$?

In [ ]:
n_glass, n_air = 1.5, 1.0
for angle in (30, 50):
    out = waves.refraction(np.radians(angle), n_glass, n_air)
    print(f"{angle} degrees: " + ("totally internally reflected" if out is None else f"refracted at {np.degrees(out):.1f} degrees"))

incidence = np.linspace(0, 89.9, 900)
refracted = [waves.refraction(np.radians(i), n_glass, n_air) for i in incidence]
critical = np.degrees(np.arcsin(n_air / n_glass))
plt.figure(figsize=(7, 3.6))
plt.plot(incidence, [np.nan if r is None else np.degrees(r) for r in refracted])
plt.axvspan(critical, 90, color="C3", alpha=0.15, label="total internal reflection")
plt.axvline(critical, color="C3", ls=":")
plt.annotate(f"C = {critical:.1f}°", (critical, 20), textcoords="offset points", xytext=(5, 0), color="C3")
plt.xlabel("angle of incidence in glass / degrees"); plt.ylabel("angle in air / degrees")
plt.xlim(0, 90); plt.ylim(0, 92); plt.legend(loc="upper left"); plt.title("Glass (n = 1.5) to air");

**Explain (P3).** Compare each angle with the critical angle. What does `None` mean physically?

**Optical fibres** use total internal reflection. A glass core is surrounded by a **cladding** of slightly lower refractive index. Light that meets the boundary at more than the critical angle stays trapped in the core. The cladding protects the core and stops light leaking into a neighbouring fibre that touches it. A narrow core with a refractive index close to the cladding's reduces **modal dispersion**, because only rays nearly parallel to the axis are trapped, so they all take nearly the same time.

## 5. Thin lenses (extension)

A converging lens of focal length $f$ forms an image at distance $v$ from an object at distance $u$, where (with real distances positive)

$$\frac{1}{f} = \frac{1}{u} + \frac{1}{v}, \qquad \text{magnification } m = \frac{v}{u}.$$

When the object is beyond the focal point the image is real and inverted. When it is inside the focal point, $v$ comes out negative: the image is **virtual**, on the same side as the object, as in a magnifying glass.

In [ ]:
f = 0.10
u = np.linspace(0.02, 0.50, 500)
u = u[np.abs(u - f) > 0.004]                     # skip the object at the focal point (image at infinity)
v = np.array([waves.image_distance(f, x) for x in u])
plt.figure(figsize=(7, 3.6))
plt.plot(u[u > f], v[u > f], label="real image (v > 0)")
plt.plot(u[u < f], v[u < f], label="virtual image (v < 0)")
plt.axvline(f, color="0.6", ls=":"); plt.axhline(0, color="0.7", lw=0.6)
plt.ylim(-1.0, 1.0); plt.xlabel("object distance u / m"); plt.ylabel("image distance v / m")
plt.legend(); plt.title("Converging lens, f = 0.10 m")
for x in (0.30, 0.20, 0.05):
    image = waves.image_distance(f, x)
    print(f"u = {x:.2f} m: v = {image:+.3f} m, magnification {image / x:+.2f}")

At $u = 2f$ the image is also at $2f$ and the same size. A negative $v/u$ marks a virtual, upright image: at $u = 0.05$ m it is twice the size of the object.

**Common mistakes**

- Using lines per mm as the spacing: $d$ is **one over** the number of lines per metre.
- Measuring angles in Snell's law from the surface instead of the normal.
- Forgetting the zero order, or one side, when counting the total number of maxima.
- Writing the single-slit minimum as $a\sin\theta = n\lambda$ and then calling it a maximum: for a single slit this condition gives the **dark** fringes.

---

## Exercises

Give answers to 3 significant figures.

**Q1.** A grating has 300 lines per mm. The second-order maximum for a spectral line is at $20.5^\circ$. Find the wavelength.

**Q2.** Light of wavelength 600 nm falls on a grating with 500 lines per mm. Find the total number of bright maxima, including the zero order.

**Q3.** An optical fibre has a core of refractive index 1.48 and a cladding of 1.46. Find the critical angle at the core–cladding boundary, in degrees.

**Q4.** Light of wavelength 500 nm passes through a single slit 10 μm wide onto a screen 2.0 m away. Find the width of the central maximum on the screen (between the first minima on either side).

**Q5.** An object is 0.30 m from a converging lens of focal length 0.10 m. Find the image distance.

**Q6 (simulation).** Light of wavelength 500 nm falls on a grating of spacing 1.2 μm. Use `waves.grating_angle` in a loop to find the highest order that exists, then give its angle in degrees.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_answers = {
    "Q1": (1e-3 / 300) * np.sin(np.radians(20.5)) / 2,
    "Q2": 2 * int(np.floor((1e-3 / 500) / 600e-9)) + 1,
    "Q3": np.degrees(np.arcsin(1.46 / 1.48)),
    "Q4": 2 * 2.0 * np.tan(np.arcsin(500e-9 / 10e-6)),
    "Q5": waves.image_distance(0.10, 0.30),
    "Q6": np.degrees(max((waves.grating_angle(n, 500e-9, 1.2e-6), n) for n in range(1, 10)
                         if waves.grating_angle(n, 500e-9, 1.2e-6) is not None)[0]),
}

# Replace None with your answers, in SI units unless the question says otherwise.
mine = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** $d = 1/(300\times10^{3}) = 3.33\times10^{-6}$ m. Then $\lambda = \dfrac{d\sin\theta}{n} = \dfrac{3.33\times10^{-6} \times \sin 20.5^\circ}{2} = 5.84\times10^{-7}$ m, about 584 nm (the yellow sodium line).

**Q2.** $d = 1/(500\times10^{3}) = 2.0\times10^{-6}$ m, so $d/\lambda = 2.0\times10^{-6}/600\times10^{-9} = 3.33$. The highest order is $n = 3$. Counting both sides and the zero order: $2 \times 3 + 1 = 7$ maxima.

**Q3.** $\sin C = n_2/n_1 = 1.46/1.48$, so $C = 80.6^\circ$. Only rays nearly parallel to the fibre's axis are trapped.

**Q4.** $\sin\theta = \lambda/a = 500\times10^{-9}/10\times10^{-6} = 0.050$, so $\theta = 2.87^\circ$. Each first minimum is $2.0\tan\theta = 0.100$ m from the centre, so the central maximum is $0.200$ m wide.

**Q5.** $\dfrac{1}{v} = \dfrac{1}{f} - \dfrac{1}{u} = \dfrac{1}{0.10} - \dfrac{1}{0.30} = 6.67\ \mathrm{m^{-1}}$, so $v = 0.150$ m. The image is real, inverted and half the size ($m = v/u = 0.500$).

**Q6.** $d/\lambda = 1.2\times10^{-6}/500\times10^{-9} = 2.4$, so the highest order is 2 and order 3 would need $\sin\theta = 1.25$. Order 2 is at $\sin\theta = 2 \times 500\times10^{-9}/1.2\times10^{-6} = 0.833$, so $\theta = 56.4^\circ$. The loop below agrees.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}".rstrip("."))

_orders = [n for n in range(1, 10) if waves.grating_angle(n, 500e-9, 1.2e-6) is not None]
print(f"orders that exist: {_orders}; order {_orders[-1]} at {np.degrees(waves.grating_angle(_orders[-1], 500e-9, 1.2e-6)):.2f} degrees")

### Predict and explain: answers

**P1.** The positions stay the same: they depend only on $d$ and $\lambda$ through $d\sin\theta = n\lambda$, so every pattern has its maxima on the same dotted lines. The maxima get much **narrower** (and, for the same light per slit, much brighter, since the amplitudes of $N$ slits add). With 2 slits the light drops off gradually either side of a maximum. With 20 slits a tiny move off the maximum puts the first and last slits half a wavelength or more out of step, and the waves from all the slits largely cancel. That is why gratings are used to measure wavelengths precisely.

**P2.** Yes. Second-order red (700 nm) is at about $24.8^\circ$, but third-order violet (400 nm) is at only $21.1^\circ$, so the start of the third-order spectrum lies inside the second. Third-order light of wavelength $\tfrac{2}{3} \times 700 = 467$ nm lands exactly on second-order red, because $3 \times 467 = 2 \times 700$.

**P3.** $50^\circ$. The critical angle is $\sin^{-1}(1/1.5) = 41.8^\circ$. At $30^\circ$, below the critical angle, the light refracts out at $48.6^\circ$. At $50^\circ$, above it, Snell's law would need $\sin\theta_2 = 1.15$, which is impossible: there is no refracted ray (`None`), and all the light reflects back into the glass.